# Project 3: Load shipments into a free cloud MySQL database (Aiven)

Before running: your Aiven MySQL service must show **Running**. Run the upload cell and choose BOTH `shipments_clean.csv` (from Project 2) and `ca.pem` (the CA certificate from the Aiven Overview page).

In [2]:
# Upload shipments_clean.csv AND ca.pem (select both)
from google.colab import files
uploaded = files.upload()

Saving ca.pem to ca.pem


## [1] Install the MySQL driver

In [3]:
!pip install -q pymysql sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.4 MB/s eta 0:00:00


## [2] Connect (copy these from the Aiven service Overview page)

In [ ]:
import os, pandas as pd
from getpass import getpass
from sqlalchemy import create_engine, text

HOST = os.environ.get("DB_HOST") or input("Host: ").strip()
PORT = int(os.environ.get("DB_PORT") or input("Port: ").strip())
USER = os.environ.get("DB_USER") or input("User (usually avnadmin): ").strip()
PASSWORD = os.environ.get("DB_PASS") or getpass("Password: ")

# Aiven requires an encrypted (SSL) connection; ca.pem is the certificate you uploaded
ssl_args = {"ssl": {"ca": "ca.pem"}} if os.path.exists("ca.pem") else {}
server = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}", connect_args=ssl_args)
with server.connect() as conn:
    print("Connected to MySQL", conn.execute(text("SELECT VERSION()")).scalar())

## [3] Create the database

In [5]:
with server.begin() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS supply_chain"))
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/supply_chain", connect_args=ssl_args)

## [4] Design the schema: two related tables

In [8]:
schema = """
DROP TABLE IF EXISTS shipments;
DROP TABLE IF EXISTS carriers;

CREATE TABLE carriers (
    carrier_id   INT AUTO_INCREMENT PRIMARY KEY,
    carrier_name VARCHAR(50) NOT NULL UNIQUE
);

CREATE TABLE shipments (
    shipment_id      VARCHAR(10) PRIMARY KEY,
    carrier_id       INT NOT NULL,
    mode             VARCHAR(10) NOT NULL,
    origin           VARCHAR(50),
    destination      VARCHAR(50),
    order_date       DATE NOT NULL,
    ship_date        DATE NOT NULL,
    delivery_date    DATE NULL,
    promised_days    INT NOT NULL,
    weight_kg        DECIMAL(10,1) CHECK (weight_kg > 0),
    freight_cost_usd DECIMAL(10,2) CHECK (freight_cost_usd >= 0),
    FOREIGN KEY (carrier_id) REFERENCES carriers(carrier_id)
);
"""
with engine.begin() as conn:
    for statement in schema.split(";"):
        if statement.strip():
            conn.execute(text(statement))
print("Tables created")

Tables created


## [5] Load the cleaned data from Project 2

In [9]:
df = pd.read_csv("shipments_clean.csv")

for col in ["order_date", "ship_date", "delivery_date"]:
    df[col] = pd.to_datetime(df[col], format="mixed").dt.date

carriers = pd.DataFrame({"carrier_name": sorted(df["carrier"].unique())})
carriers.to_sql("carriers", engine, if_exists="append", index=False)

carrier_ids = pd.read_sql("SELECT carrier_id, carrier_name FROM carriers", engine)
ship = df.merge(carrier_ids, left_on="carrier", right_on="carrier_name")
cols = ["shipment_id", "carrier_id", "mode", "origin", "destination", "order_date",
        "ship_date", "delivery_date", "promised_days", "weight_kg", "freight_cost_usd"]
ship[cols].to_sql("shipments", engine, if_exists="append", index=False)

1200

## [6] Validate the load

In [10]:
print("Rows in CSV:  ", len(df))
print("Rows in MySQL:", pd.read_sql("SELECT COUNT(*) AS n FROM shipments", engine)["n"][0])

Rows in CSV:   1200
Rows in MySQL: 1200


## [7] Query 1: late rate by carrier (JOIN + GROUP BY)

In [11]:
q1 = """
SELECT c.carrier_name,
       COUNT(*) AS shipments,
       ROUND(100 * AVG(DATEDIFF(s.delivery_date, s.ship_date) > s.promised_days), 1) AS pct_late
FROM shipments s
JOIN carriers c ON c.carrier_id = s.carrier_id
WHERE s.delivery_date IS NOT NULL
GROUP BY c.carrier_name
ORDER BY pct_late DESC;
"""
print(pd.read_sql(q1, engine))

          carrier_name  shipments  pct_late
0  Northstar Logistics        240      32.5
1        Atlas Freight        244      14.8
2       Harbor Express        258      14.7
3    BlueWave Shipping        242      12.8
4     Summit Air Cargo        196      10.7


## [8] Query 2: monthly volume and spend

In [12]:
q2 = """
SELECT DATE_FORMAT(order_date, '%%Y-%%m') AS month,
       COUNT(*)                         AS shipments,
       ROUND(SUM(freight_cost_usd), 0)  AS total_spend_usd
FROM shipments
GROUP BY month
ORDER BY month;
"""
print(pd.read_sql(q2, engine))

     month  shipments  total_spend_usd
0  2026-01        121         183185.0
1  2026-02        145         229934.0
2  2026-03        170         259638.0
3  2026-04        146         225286.0
4  2026-05        166         261378.0
5  2026-06        154         247227.0
6  2026-07        155         223603.0
7  2026-08        143         214441.0


## [9] Query 3: busiest routes, only those with 10+ shipments (HAVING)

In [13]:
q3 = """
SELECT origin, destination, COUNT(*) AS shipments,
       ROUND(AVG(freight_cost_usd), 0) AS avg_cost_usd
FROM shipments
GROUP BY origin, destination
HAVING COUNT(*) >= 10
ORDER BY shipments DESC
LIMIT 5;
"""
print(pd.read_sql(q3, engine))

      origin destination  shipments  avg_cost_usd
0   Shanghai     Atlanta         41        1921.0
1   Shanghai     Chicago         41        1456.0
2   Montreal     Buffalo         40        1825.0
3  Rotterdam   Rochester         39        1957.0
4   Shanghai   Rochester         39        1791.0


## [10] Query 4: label each shipment (CASE)

In [14]:
q4 = """
SELECT shipment_id, mode,
       DATEDIFF(delivery_date, ship_date) - promised_days AS days_over,
       CASE
           WHEN delivery_date IS NULL THEN 'In transit / unknown'
           WHEN DATEDIFF(delivery_date, ship_date) <= promised_days THEN 'On time'
           WHEN DATEDIFF(delivery_date, ship_date) - promised_days <= 2 THEN 'Slightly late'
           ELSE 'Very late'
       END AS status
FROM shipments
ORDER BY days_over DESC
LIMIT 10;
"""
print(pd.read_sql(q4, engine))

  shipment_id   mode  days_over     status
0     SH10093  Truck          6  Very late
1     SH10111  Ocean          6  Very late
2     SH10144  Ocean          6  Very late
3     SH10242  Ocean          6  Very late
4     SH10199  Ocean          6  Very late
5     SH10140  Ocean          6  Very late
6     SH10109  Truck          6  Very late
7     SH10105  Truck          6  Very late
8     SH10195  Truck          6  Very late
9     SH10084  Ocean          6  Very late


## [11] Test a rule: the database should reject a bad row

In [15]:
try:
    with engine.begin() as conn:
        conn.execute(text("""INSERT INTO shipments VALUES
            ('BAD001', 999, 'Truck', 'X', 'Y', '2026-01-01', '2026-01-02', NULL, 5, 100, 50)"""))
except Exception as e:
    print("Rejected, as it should be:", type(e).__name__)

Rejected, as it should be: IntegrityError
